# transformer_ko4 — knockout-context transformer with spatial gating (Colab / GPU)

**v4** of the knockout far-field transformer: a knockout is a *set of tokens* (perturbed gene + its post-KO co-dependency /
complex / PPI partners), each carrying `[DepMap | network-SVD | essentiality | role]` + **physics edge-features** (interface
n_pdb, co-dependency magnitude, shared-complex) + a **Graphormer edge-bias** + **spatial gating** (GO compartment: same-
compartment attention bias / hard mask). Trained as a learned-retrieval metric, scored on held-out K562 Perturb-seq
(tide-removed specific-mover recall@50) vs a tide-null floor and the retrieval oracle.

**Honest expectation:** the gap to the ~0.62 oracle is data/readout-limited (a same-KO cross-line probe showed the specific
far-field is ~85% context-specific), so spatial gating is expected to be *marginal* on 96h steady-state mRNA. This is the honest
test — and a GPU base to extend toward the GPU-gated layers (AF-Multimer, ΔΔG).

### Data
`nlz_K562.pkl` now ships in the repo (small). You still need two large artifacts in Google Drive under
`MyDrive/cell_model/` (or `.../artifacts`): **`cell_complete.json`** and **`depmap_vecs.npz`** (the project persists these via
`persist.py`). Set **Runtime → Change runtime type → GPU**.


In [ ]:
# 1. deps (Colab already has torch+CUDA, numpy, scipy)
import torch
print('torch', torch.__version__, '| CUDA', torch.cuda.is_available(), '|', torch.cuda.get_device_name(0) if torch.cuda.is_available() else 'CPU')
!pip -q install scipy >/dev/null 2>&1


In [ ]:
# 2. clone the repo (or update it to the latest branch head so nlz_K562.pkl is present)
import os
BRANCH = 'claude/vectorize-gex-propensity-zp09w8'
if not os.path.isdir('/content/cell'):
    !git clone --branch $BRANCH --depth 1 https://github.com/Nikku03/cell.git /content/cell
else:
    !cd /content/cell && git fetch --depth 1 origin $BRANCH && git reset --hard origin/$BRANCH
%cd /content/cell
!ls -la outputs/orphan/nlz_K562.pkl colab/transformer_ko4.py && echo OK


In [ ]:
# 3. mount Drive and stage the artifacts into the paths the code expects
from google.colab import drive
import glob, shutil, os
drive.mount('/content/drive')
SP = '/tmp/claude-0/-home-user-cell/0f039315-b3a9-52ac-8187-9fae0d726994/scratchpad'   # eval_harness looks here for nlz_K562.pkl
os.makedirs(SP, exist_ok=True); os.makedirs('/content/cell/outputs/orphan', exist_ok=True)
NEED = {'cell_complete.json':'/content/cell/outputs/orphan/cell_complete.json',
        'depmap_vecs.npz':'/content/cell/outputs/orphan/depmap_vecs.npz',
        'nlz_K562.pkl': SP + '/nlz_K562.pkl'}
# search the cloned repo FIRST (nlz ships there), then Drive (the two big files)
SEARCH = ['/content/cell/outputs/orphan','/content/drive/MyDrive/cell_model/artifacts',
          '/content/drive/MyDrive/cell_model/caches','/content/drive/MyDrive/cell_model',
          '/content/drive/MyDrive/nexus_cache','/content/drive/MyDrive']
def find(name):
    for d in SEARCH:
        hits = glob.glob(os.path.join(d, '**', name), recursive=True)
        if hits: return hits[0]
    return None
missing = []
for name, dst in NEED.items():
    if os.path.exists(dst) and os.path.getsize(dst) > 0: print('present:', name); continue
    src = find(name)
    if src and os.path.abspath(src) != os.path.abspath(dst): shutil.copy(src, dst); print(f'copied {name}  <-  {src}')
    elif src: print('present (in place):', name)
    else: missing.append(name)
assert not missing, ('MISSING in Drive: ' + ', '.join(missing) +
    '  -> put them under MyDrive/cell_model/ (artifacts or caches) and re-run this cell.')
print('all artifacts staged.')


In [ ]:
# 4. (optional) fast end-to-end smoke check on GPU: 1 seed, reduced configs/epochs (~1-2 min on GPU)
!cd /content/cell && V4_SMOKE=1 python colab/transformer_ko4.py


In [ ]:
# 5. FULL run: 3 splits x {v3-full base, +spatial-bias, +spatial-mask} + controls, GPU
!cd /content/cell && python colab/transformer_ko4.py


### Reading the output
- **`v3-full (no spatial)`** = base (edge-features + Graphormer edge-bias, no spatial).
- **`+spatial-bias` / `+spatial-mask`** add the GO-compartment attention bias / hard co-localization gate.
- **`spatial - base`** is the number that matters; **`TIDE-null`** (~0.25) is the floor, **`ORACLE*`** (~0.61) the retrieval ceiling,
  **`wrong-KO shuffle`** the identity control (should collapse).
- ~0 (the honest prior) ⇒ localization already latent in the network-SVD, ceiling is data-limited. A stable positive with the shuffle
  control passing ⇒ spatial gating genuinely helps.

To break past the retrieval ceiling you'd need **transient (2–12h) kinetics** data; the architecture is ready — swap the target matrix.
